# 04 — BERT Cased: Casing Strategy Ablation
Train `bert-base-cased` with identical config to the uncased baseline and evaluate on clean + noisy data.

In [1]:
import sys, os, glob
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from train import tokenize_and_align_labels, make_compute_metrics

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

MODEL_NAME = "bert-base-cased"
RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "results")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
CHECKPOINTS_DIR = os.path.join(RESULTS_DIR, "checkpoints")
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "noisy")
os.makedirs(TABLES_DIR, exist_ok=True)

Device: mps


In [2]:
dataset = load_dataset("conll2003", trust_remote_code=True)
label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
num_labels = len(label_names)
print(f"Labels ({num_labels}): {label_names}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

Labels (9): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Train: 14041 | Val: 3250 | Test: 3453


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model = BertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
print(f"Loaded {MODEL_NAME}")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

Loaded bert-base-cased


In [4]:
tokenized = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer),
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Tokenized. Example token count:", len(tokenized["train"][0]["input_ids"]))

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokenized. Example token count: 512


In [5]:
training_args = TrainingArguments(
    output_dir=os.path.join(CHECKPOINTS_DIR, MODEL_NAME.replace("/", "_")),
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=make_compute_metrics(label_names),
)

trainer.train()

/Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.050208,0.042964,0.930766,0.926037,0.935544


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=878, training_loss=0.11823953908776912, metrics={'train_runtime': 1599.5853, 'train_samples_per_second': 8.778, 'train_steps_per_second': 0.549, 'total_flos': 3669099951393792.0, 'train_loss': 0.11823953908776912, 'epoch': 1.0})

## Evaluate on clean test set

In [6]:
trainer.callback_handler.on_train_begin(trainer.args, trainer.state, trainer.control)
trainer.eval_dataset = tokenized["test"]
clean_results = trainer.evaluate()
print(f"BERT-cased (clean) | F1: {clean_results['eval_f1']:.4f} | "
      f"Precision: {clean_results['eval_precision']:.4f} | "
      f"Recall: {clean_results['eval_recall']:.4f}")

/Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.050208,0.090916,0.896515,0.888947,0.904214


BERT-cased (clean) | F1: 0.8965 | Precision: 0.8889 | Recall: 0.9042


## Evaluate on noisy test sets

In [7]:
def evaluate_on_noisy(noise_type):
    noisy_ds = load_from_disk(os.path.join(DATA_DIR, noise_type))
    tokenized_noisy = noisy_ds["test"].map(
        lambda ex: tokenize_and_align_labels(ex, tokenizer),
        batched=True,
        remove_columns=noisy_ds["test"].column_names,
    )
    trainer.callback_handler.on_train_begin(trainer.args, trainer.state, trainer.control)
    trainer.eval_dataset = tokenized_noisy
    metrics = trainer.evaluate()
    return {
        "model": MODEL_NAME,
        "noise": noise_type,
        "f1": round(metrics["eval_f1"], 4),
        "precision": round(metrics["eval_precision"], 4),
        "recall": round(metrics["eval_recall"], 4),
    }

In [8]:
noisy_results = []
for noise_type in ["ocr", "asr", "social"]:
    print(f"Evaluating on {noise_type}...")
    noisy_results.append(evaluate_on_noisy(noise_type))

noisy_df = pd.DataFrame(noisy_results)
print(noisy_df)

Evaluating on ocr...


Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.050208,0.322690,0.699821,0.732597,0.669852


Evaluating on asr...


Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.050208,1.391217,0.213418,0.582019,0.130666


Evaluating on social...


Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.050208,0.252331,0.709597,0.668545,0.756020


             model   noise      f1  precision  recall
0  bert-base-cased     ocr  0.6998     0.7326  0.6699
1  bert-base-cased     asr  0.2134     0.5820  0.1307
2  bert-base-cased  social  0.7096     0.6685  0.7560


## Summary: Clean + Noisy

In [9]:
# Baseline from notebook 01 (bert-base-uncased)
uncased_baseline = {
    "ocr": 0.6632, "asr": 0.8255, "social": 0.6174, "clean": 0.890836
}

rows = [{"noise": "clean", "f1_cased": clean_results["eval_f1"],
         "f1_uncased": uncased_baseline["clean"]}]
for r in noisy_results:
    rows.append({
        "noise": r["noise"],
        "f1_cased": r["f1"],
        "f1_uncased": uncased_baseline[r["noise"]],
    })

summary_df = pd.DataFrame(rows)
summary_df["delta_cased_vs_uncased"] = (summary_df["f1_cased"] - summary_df["f1_uncased"]).round(4)
print(summary_df.to_string(index=False))

 noise  f1_cased  f1_uncased  delta_cased_vs_uncased
 clean  0.896515    0.890836                  0.0057
   ocr  0.699800    0.663200                  0.0366
   asr  0.213400    0.825500                 -0.6121
social  0.709600    0.617400                  0.0922


In [10]:
# Save noisy results
noisy_df["f1_baseline"] = noisy_df["noise"].map({
    "ocr": clean_results["eval_f1"],
    "asr": clean_results["eval_f1"],
    "social": clean_results["eval_f1"],
})
noisy_df["f1_delta"] = (noisy_df["f1"] - noisy_df["f1_baseline"]).round(4)
noisy_df["f1_drop_pct"] = (noisy_df["f1_delta"] / noisy_df["f1_baseline"] * 100).round(2)

noisy_df.to_csv(os.path.join(TABLES_DIR, "bert_cased_noisy_results.csv"), index=False)

# Save clean result
clean_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "strategy": "WordPiece (cased)",
    "f1_clean": clean_results["eval_f1"],
    "precision_clean": clean_results["eval_precision"],
    "recall_clean": clean_results["eval_recall"],
}])
clean_df.to_csv(os.path.join(TABLES_DIR, "bert_cased_clean_results.csv"), index=False)

print("Saved bert_cased_noisy_results.csv and bert_cased_clean_results.csv")
noisy_df

Saved bert_cased_noisy_results.csv and bert_cased_clean_results.csv


,model,noise,f1,precision,recall,f1_baseline,f1_delta,f1_drop_pct
0,bert-base-cased,ocr,0.6998,0.7326,0.6699,0.896515,-0.1967,-21.94
1,bert-base-cased,asr,0.2134,0.5820,0.1307,0.896515,-0.6831,-76.20
2,bert-base-cased,social,0.7096,0.6685,0.7560,0.896515,-0.1869,-20.85
